# Writing an HDF5 loading backend

This notebook implements a small backend interface used by `labdata.load`. It demonstrates:

- discovering variables and images in a laboratory HDF5 schema;
- reading unit metadata and returning values in configured output units;
- exposing a derived `OD` image whenever `ABS` and `REF` are present;
- attaching the backend to the ordinary `list_file`, `load_file`, and `load_data` functions.

## Import

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np

from labdata import config, load
from labdata.matplotlib import UNITS

%load_ext labdata.notebook

## File layout

Each example file has the same small HDF5 structure. Units belong to the datasets that store the values.

```text
demo41.h5
├── vars
│   ├── duration
│   │   └── @unit = "us"
│   └── detuning
│       └── @unit = "kHz"
└── data
    └── image
        └── CAM_VERTICAL
            ├── abs    (64 × 64)
            └── ref    (64 × 64)
```

`OD` is not stored in the file; the backend advertises and constructs it from `abs` and `ref`.

In [ ]:
PATH = Path("data")

## Backend selection and units

A backend is an ordinary module. `backend` validates the schema and returns that module; this is also where a lab package can dispatch between multiple file formats. Unit conversion reuses the same SI-prefix definitions as LabData's Matplotlib formatter.

In [ ]:
CAMERA = "CAM_VERTICAL"


def _text(value):
    return value.decode() if isinstance(value, bytes) else str(value)


def _field(name):
    match = re.fullmatch(r"(.+?)(?:\[([^]]+)\])?", name)
    return match.groups()


def _convert(value, source, target):
    if not source or not target or source == target:
        return value
    source_scale, source_base, _ = UNITS[source]
    target_scale, target_base, _ = UNITS[target]
    if source_base != target_base:
        raise ValueError(f"Cannot convert {source} to {target}")
    return value * source_scale / target_scale


def backend(file):
    if "vars" not in file or f"data/image/{CAMERA}" not in file:
        raise ValueError("Unknown HDF5 schema")
    return sys.modules[__name__]

## Variables

`list_var` publishes exact field names using the units stored in HDF5. `load_var` reads the requested unit from its field name and converts the stored value before returning it, so the same dataset can be requested as `duration[us]`, `duration[ms]`, or `duration[s]`. The unit in brackets is authoritative: it describes the unit in which the returned number is stored and must be interpreted, rather than being only a display label.

In [ ]:
def list_var(file):
    fields = []
    for name, dataset in file["vars"].items():
        stored_unit = dataset.attrs.get("unit")
        stored_unit = _text(stored_unit) if stored_unit else None
        fields.append(f"{name}[{stored_unit}]" if stored_unit else name)
    return fields


def load_var(file, name):
    field, requested_unit = _field(name)
    dataset = file[f"vars/{field}"]
    stored_unit = dataset.attrs.get("unit")
    stored_unit = _text(stored_unit) if stored_unit else None
    value = dataset[()]
    value = value.item() if isinstance(value, np.generic) else value
    return _convert(value, stored_unit, requested_unit)

In [ ]:
with load.open_file((PATH / "demo41.h5").resolve()) as file:
    print("duration[us] =", load_var(file, "duration[us]"))
    print("duration[ms] =", load_var(file, "duration[ms]"))
    print("duration[s]  =", load_var(file, "duration[s]"))

## Images

`list_img` reports images as `CAMERA//NAME` and adds `CAM_VERTICAL//OD` only when both inputs needed to construct it exist. `load_img` handles stored and derived names uniformly, including optional ROI slicing in `(row_start, row_stop, column_start, column_stop)` order.

In [ ]:
def list_img(file):
    names = [name.upper() for name in file[f"data/image/{CAMERA}"]]
    if {"ABS", "REF"} <= set(names):
        names.append("OD")
    return [f"{CAMERA}{config.SEPARATOR}{name}" for name in names]


def load_img(file, name, roi=None):
    camera, name = name.split(config.SEPARATOR, 1)
    name = name.upper()
    if name == "OD":
        absorption = load_img(file, f"{camera}{config.SEPARATOR}ABS")
        reference = load_img(file, f"{camera}{config.SEPARATOR}REF")
        image = -np.log(np.clip(absorption / reference, 1e-12, None))
    else:
        image = np.asarray(file[f"data/image/{camera}/{name.lower()}"], dtype=float)
    if roi is not None:
        row_start, row_stop, column_start, column_stop = roi
        image = image[row_start:row_stop, column_start:column_stop]
    return image

## Metadata

The final two backend functions define the file- and image-level metadata retained by `load_file` and `load_data`.

In [ ]:
def meta_file(file):
    return {
        "FILE": file.source,
        "METADATA": {"CAMERA": CAMERA},
    }


def meta_img(file, image):
    camera, name = image.split(config.SEPARATOR, 1)
    return {
        "CAMERA": camera,
        "NAME": name.upper(),
    }

## Configuration

An installed lab package normally provides these settings in its `config.py`. Assigning them here keeps the example self-contained while using exactly the same loading API.

In [ ]:
config.DATA_FOLDER = Path(".").resolve()
config.DATA_FILE_PATTERN = r"(?i).*\.h5$"
config.LOAD_BACKEND = backend

In [ ]:
FILE = load.list_file(".")

print(FILE)
print(load.list_var(FILE[0]))
print(load.list_img(FILE[0]))

## Loading

The generic loader now sees the derived image and converted fields like any other backend output. `load_file` loads everything; `load_data` selects one analysis image, variables, and an optional ROI.

In [ ]:
d = load.load_file(FILE)

In [ ]:
%print d

In [ ]:
VAR = ["duration[ms]", "detuning[MHz]"]
DATA = load.load_data(FILE, var=VAR, img="CAM_VERTICAL//OD", roi=(8, 56, 8, 56))

%print DATA

In [ ]:
DATA.p.table(x="duration[ms]", width=100)
DATA.p.imshow(vmin=0, vmax=1.4, cmap="viridis")

In a real lab extension, these backend definitions live in `load.py`, while `config.py` sets `LOAD_BACKEND = load.backend` together with the data folder and filename pattern. The shared package continues to own path handling, vectorized loading, DataFrame construction, progress, and consolidated collection errors.

## Typical notebook workflow

With the backend installed, a working analysis notebook does not contain any of its implementation. An installed lab package normally supplies the data root; for this self-contained notebook, point the configuration at the demo directory explicitly.

In [ ]:
config.DATA_FOLDER = Path(".").resolve()
config.DEFAULT_IMAGE = "OD"

Now `%view` finds `data` as its only data folder. It can browse the demo HDF5 files, inspect the available variables and images, and select an ROI.

In [ ]:
%view

Then keep the resulting choices together near the top of the analysis, as in the laboratory notebooks:

In [ ]:
PATH = "data"  # relative to DATA_FOLDER
VAR = [
    "duration[ms]",
    "detuning[MHz]",
]
IMG = "CAM_VERTICAL//OD"
ROI = (8, 56, 8, 56)
EXCLUDE = []

FILE = load.list_file(PATH, exc=EXCLUDE)
DATA = load.load_data(FILE, var=VAR, img=IMG, roi=ROI)

In [ ]:
%print DATA